In [1]:
#!/usr/bin/env python3
# =============================================================================
# RAPIERNET-RDD2022  --  Beats StripRFNet target build
# -----------------------------------------------------------------------------
# Goal: outperform StripRFNet (mAP50 84.4 / mAP50:95 52.0 on China subset,
#       F1 80.33 on full RDD2022) using a yolo11m-scale RapierNet
#       (SARF + CoordGate + DySample) with correct RDD2022 4-class labels,
#       an honest train/val split, high-res fine-tune, and WBF+TTA inference.
#
# KEY FIX vs previous runs:
#   * The Kaggle dump RDD_SPLIT has 5 class IDs [0..4] but uses the order
#     [0]=alligator(D20), [1]=transverse(D10), [2]=longitudinal(D00),
#     [3]=other corruption(D40), [4]=Pothole(dup of D40).
#   * RDD2022 officially has 4 classes (D00/D10/D20/D40). Class 4 "Pothole"
#     duplicates D40, so we REMAP to official 4 classes and MERGE id 4 into D40.
#   * The dump's train/val/test are an overlapping split of the same source, so
#     we build our OWN leakage-free train/val split for an honest comparison.
#
# RESUME / 12-HR SESSION LIMIT + COCO INIT:
#   * Stage-1 now starts from a COCO-pretrained yolo11m backbone (PRETRAINED),
#     copied layer-by-layer where shapes match; SARF/CoordGate/DySample + the
#     rewired Detect head keep their own (random) init. This is the key boost
#     vs the earlier random-init run (mAP50 0.524).
#   * Trainer saves checkpoints every SAVE_PERIOD epochs (default 10).
#   * At session end, run pack_next_session() -> /kaggle/working/rapiernet_continue.zip.
#   * Upload that zip as a Kaggle dataset; next session auto-restores it into
#     /kaggle/working via restore_from_input() and resumes from last.pt.
# =============================================================================

import os, sys, gc, math, random, copy, json, time, glob, shutil, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

warnings.filterwarnings("ignore")
random.seed(42); np.random.seed(42); torch.manual_seed(42)

# ------------------------------- PATH CONFIG --------------------------------
DATASET      = "/kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT"
WORKDIR      = "/kaggle/working"
DATA_NAME    = "rapiernet"                                  # run name
MODEL_SCALE  = "m"          # backbone scale n/s/m/l/x  (m = bigger than StripRFNet's s)
EPOCHS_STAGE1 = 180         # main 640px training
EPOCHS_STAGE2 = 30          # high-res 960px fine-tune
SAVE_PERIOD   = 10          # checkpoint cadence for 12-hr resume
PRETRAINED    = "yolo11m.pt"   # COCO init for stage-1 (None = random init)
os.makedirs(WORKDIR, exist_ok=True)

# Offline setting fixes headless validation
os.environ["WANDB_SILENT"] = "true"
os.environ["YOLO_OFFLINE"] = "1"

print("="*72)
print("  RAPIERNET-RDD2022  (scale=%s)  ->  target: beat StripRFNet" % MODEL_SCALE)
print("="*72)
print("Python", sys.version.split()[0], "| torch", torch.__version__,
      "| CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    seen = {}
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        mem  = torch.cuda.get_device_properties(i).total_memory/1e9
        k = (name, round(mem,1))
        seen[k] = seen.get(k,0)+1
    for (n,m),cnt in seen.items():
        print("  GPU: %s  %.1f GB  x%d" % (n,m,cnt))
    torch.backends.cudnn.benchmark = True

try:
    import ultralytics
    print("ultralytics", ultralytics.__version__)
    from ultralytics import YOLO
except Exception:
    os.system("pip install -q ultralytics albumentations ensemble-boxes -U")
    import ultralytics
    from ultralytics import YOLO
try:
    import ensemble_boxes; HAS_WBF=True
except Exception:
    HAS_WBF=False

# ========================== CUSTOM RAPIERNET MODULES ==========================
class DeformableStripConv(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.k=k; self.offset=None; self.conv_h=None; self.conv_v=None; self.gate=None; self.act=nn.GELU()
    def _init(self,c,device):
        if self.conv_h is None or self.conv_h.in_channels!=c:
            k,pad=self.k,self.k//2
            self.conv_h=nn.Conv2d(c,c,(1,k),padding=(0,pad),groups=c,bias=False,device=device)
            self.conv_v=nn.Conv2d(c,c,(k,1),padding=(pad,0),groups=c,bias=False,device=device)
            self.offset=nn.Conv2d(c,2,3,padding=1,bias=True,device=device)
            self.gate=nn.Conv2d(2*c,c,1,bias=False,device=device)
            self.act=nn.GELU()
    def forward(self,x):
        c=x.shape[1]; self._init(c,x.device)
        h,w=x.shape[2],x.shape[3]
        off=self.offset(x).tanh()*1.5; b=x.shape[0]
        yy,xx=torch.meshgrid(torch.linspace(-1,1,h,device=x.device),
                             torch.linspace(-1,1,w,device=x.device),indexing="ij")
        grid=torch.stack([xx,yy],-1).unsqueeze(0).expand(b,h,w,2)
        grid=grid+2.0/h*off.permute(0,2,3,1)
        xs=F.grid_sample(x,grid,mode="bilinear",align_corners=False,padding_mode="border")
        hf=self.act(self.conv_h(xs)); vf=self.act(self.conv_v(xs))
        gf=torch.sigmoid(self.gate(torch.cat([hf,vf],1)))
        return hf*gf+vf*(1-gf)

class SARF(nn.Module):
    """Shape-Adaptive Receptive Field block (channel-preserving drop-in)."""
    def __init__(self,k=7,rates=(1,3,5)):
        super().__init__(); self.k=k; self.rates=rates
        self.strip=None; self.asy=None; self.merge=None; self.bn=None; self.act=nn.SiLU()
    def _init(self,c,device):
        r=self.rates
        if self.strip is None or self.strip.gate is None or self.strip.gate.out_channels!=c:
            self.strip=DeformableStripConv(self.k); c_=max(16,c//4)
            self.asy=nn.ModuleList([nn.Conv2d(c,c_,3,padding=x,dilation=x,device=device) for x in r])
            self.merge=nn.Conv2d(c_*len(r)+c,c,1,device=device); self.bn=nn.BatchNorm2d(c,device=device)
    def forward(self,x):
        c=x.shape[1]; self._init(c,x.device)
        s=self.strip(x); a=torch.cat([m(x) for m in self.asy],1)
        return self.act(self.bn(self.merge(torch.cat([s,a],1))))+x

class CoordGate(nn.Module):
    """Spatial+channel attention gate (CBAM-style). Channel-preserving."""
    def __init__(self,r=4):
        super().__init__(); self.r=r; self.sp_conv=None; self.ch_gate=None
    def _init(self,c,device):
        if self.sp_conv is None or self.sp_conv.in_channels!=2:
            c_=max(8,c//self.r)
            self.sp_conv=nn.Conv2d(2,1,7,padding=3,device=device)
            self.ch_gate=nn.Sequential(nn.Conv2d(c,c_,1,device=device),nn.SiLU(),
                                       nn.Conv2d(c_,c,1,device=device))
    def forward(self,x):
        c=x.shape[1]; self._init(c,x.device)
        avg=x.mean(dim=1,keepdim=True); mx=x.max(dim=1,keepdim=True).values
        sp=torch.sigmoid(self.sp_conv(torch.cat([avg,mx],1))); x=x*sp
        ch=torch.sigmoid(self.ch_gate(x.mean((2,3),keepdim=True)))
        return x*ch

class DySample(nn.Module):
    """Content-adaptive dynamic upsampling (channel-preserving)."""
    def __init__(self,scale=2):
        super().__init__(); self.scale=scale; self.conv=None; self.offset=None
    def _init(self,c,device):
        if self.conv is None or self.conv.in_channels!=c:
            self.conv=nn.Conv2d(c,c,1,bias=False,device=device)
            self.offset=nn.Conv2d(c,2,3,padding=1,device=device)
    def forward(self,x):
        c=x.shape[1]; self._init(c,x.device)
        x=self.conv(x)+x; b,_,h,w=x.shape; s=self.scale
        theta=torch.eye(2,3,device=x.device).unsqueeze(0).expand(b,-1,-1)
        base=F.affine_grid(theta,(b,1,h*s,w*s),align_corners=False)
        off=F.interpolate(self.offset(x),size=(h*s,w*s),mode="bilinear",
                          align_corners=False).permute(0,2,3,1)
        return F.grid_sample(x,base+off*(0.5/max(h,w)),mode="bilinear",
                             align_corners=False,padding_mode="border")

_REGISTERED = False

def register_custom_modules():
    """Register custom blocks into ultralytics namespaces so RapierNet
    checkpoints unpickle in ANY fresh session. MUST run before any torch.load
    of a RapierNet checkpoint (e.g. seeding stage-2 from best.pt when stage-1
    is already finished and build_rapiernet is never called)."""
    global _REGISTERED
    if _REGISTERED:
        return
    import ultralytics.nn.modules as _um
    import ultralytics.nn.tasks as _ut
    for _k, _v in (("SARF", SARF), ("DeformableStripConv", DeformableStripConv),
                   ("CoordGate", CoordGate), ("DySample", DySample)):
        _v.__module__ = _um.__name__
        setattr(_um, _k, _v); setattr(_ut, _k, _v)
    _REGISTERED = True
    print("[RAPIERNET] custom modules registered (cross-session unpickle ready)")

register_custom_modules()

# ========================== STEP 0 : PREPARE CLEAN 4-CLASS DATASET ============
def remap_and_split():
    """Build a leakage-free 4-class YOLO dataset under /kaggle/working/rdd4.
    Class remap (Kaggle dump order -> official RDD2022):
        0 alligator crack     -> D20 (2)
        1 transverse crack    -> D10 (1)
        2 longitudinal crack  -> D00 (0)
        3 other corruption    -> D40 (3)
        4 Pothole (dup of D40)-> D40 (3)   [MERGED]
    Files named 'test/...' in the dump are held out entirely (they mirror train,
    so they are NOT a real held-out set). We split train/val by image id.
    """
    RDMAP = {0:2, 1:1, 2:0, 3:3, 4:3}
    src_imgs = f"{DATASET}/train/images"
    src_lbls = f"{DATASET}/train/labels"
    out = "/kaggle/working/rdd4"
    for sp in ["train/images","train/labels","val/images","val/labels"]:
        os.makedirs(os.path.join(out,sp), exist_ok=True)

    imgs = sorted(glob.glob(f"{src_imgs}/*.jpg"))
    if not imgs:
        imgs = sorted(glob.glob(f"{src_imgs}/*.png"))
    print("train images found:", len(imgs))
    rng = np.random.RandomState(42)
    rng.shuffle(imgs)
    n_val = int(len(imgs)*0.10)          # 10% val
    val_ids = set(imgs[:n_val])

    def copy_split(sp_imgs, dest_sp):
        nseg=0; nimg=0
        for im in sp_imgs:
            base=os.path.splitext(os.path.basename(im))[0]
            lbl=os.path.join(src_lbls, base+".txt")
            if not os.path.isfile(lbl):
                continue
            with open(lbl) as f:
                keep=[]
                for ln in f:
                    p=ln.split()
                    if len(p)<5: continue
                    try:
                        c=int(float(p[0]))
                    except ValueError:
                        continue
                    c = RDMAP.get(c)
                    if c is None: continue
                    keep.append(f"{c} "+" ".join(p[1:])+"\n")
            if not keep: continue
            dest_img=os.path.join(out,dest_sp,"images",os.path.basename(im))
            dest_lbl=os.path.join(out,dest_sp,"labels",base+".txt")
            shutil.copyfile(im,dest_img)
            with open(dest_lbl,"w") as f: f.writelines(keep)
            nseg+=len(keep); nimg+=1
        return nimg,nseg

    ntr,str_ = copy_split(imgs[n_val:], "train")
    nva,sva = copy_split(imgs[:n_val], "val")
    print(f"wrote train={ntr} imgs / {str_} boxes ; val={nva} imgs / {sva} boxes")

    data_yaml = "/kaggle/working/rdd4.yaml"
    with open(data_yaml,"w") as f:
        f.write("path: /kaggle/working/rdd4\n")
        f.write("train: train/images\n")
        f.write("val: val/images\n")
        f.write("nc: 4\n")
        f.write("names:\n  0: D00\n  1: D10\n  2: D20\n  3: D40\n")
    print("data yaml ->", data_yaml)
    return data_yaml, os.path.join(out,"val","images")

# ========================== STEP 1 : BUILD RAPIERNET GRAPH ====================
def build_rapiernet(scale=MODEL_SCALE, pretrained=None):
    """Register custom blocks + build a yolo11-<scale> RapierNet from the
    installed cfg. Returns a YOLO object with the custom graph baked in.
    If `pretrained` is a .pt path (e.g. 'yolo11m.pt'), COCO weights for every
    shape-compatible layer (backbone/neck/SPPF/C2PSA) are copied in; the custom
    SARF/CoordGate/DySample blocks and the rewired Detect head stay random."""
    register_custom_modules()
    import yaml, glob as _g
    parent=os.path.dirname(ultralytics.__file__)
    base_p=None
    for cand in ["11/yolo11.yaml","11/yolo11s.yaml","v8/yolov8m.yaml"]:
        p=os.path.join(parent,"cfg","models",cand)
        if os.path.exists(p): base_p=p; break
    if base_p is None:
        base_p=sorted(_g.glob(os.path.join(parent,"cfg","models","**","yolo11.yaml"),recursive=True))[0]
    cfg=yaml.safe_load(open(base_p))
    if "scales" in cfg:
        mul=cfg["scales"].get(scale, [0.5,0.5,1024])
        cfg["depth_multiple"],cfg["width_multiple"],cfg["max_channels"]=list(mul)
        cfg.pop("scales",None)
    hd=cfg["head"]
    for layer in hd:
        if isinstance(layer,list) and len(layer)>=2 and layer[2]=="nn.Upsample":
            layer[2]="DySample"; layer[3]=[2]
    blen=len(cfg["backbone"]); detect=hd[-1]
    src_abs=list(detect[0]); src_sorted=sorted(src_abs)
    new_head=list(hd[:-1])
    p4_abs=src_sorted[1]
    new_head.append([p4_abs,1,"CoordGate",[]]); coord_abs=blen+len(new_head)-1
    p5_abs=src_sorted[2]
    new_head.append([p5_abs,1,"SARF",[]]); sarf_abs=blen+len(new_head)-1
    p3_abs=src_sorted[0]
    new_head.append([list(src_abs),1,"Detect",["nc"]])
    new_head[-1][0]=[p3_abs,coord_abs,sarf_abs]
    cfg["head"]=new_head; cfg.pop("weights",None); cfg["nc"]=4
    yml=os.path.join(WORKDIR,f"rapiernet_{scale}.yaml")
    yaml.safe_dump(cfg,open(yml,"w"),sort_keys=False)
    model=YOLO(yml)
    if pretrained is not None:
        try:
            _off = os.environ.pop("YOLO_OFFLINE", None)   # let it auto-download if net
            cand = pretrained
            if not os.path.isfile(cand):
                _cands = [os.path.join(WORKDIR, os.path.basename(pretrained))]
                _cands += sorted(glob.glob("/kaggle/input/**/" + os.path.basename(pretrained),
                                             recursive=True))
                for _c in _cands:
                    if os.path.isfile(_c):
                        cand = _c; break
            try:
                from ultralytics.nn.tasks import load_checkpoint
                ck, _ckpt = load_checkpoint(cand)
            except ImportError:
                from ultralytics.nn.tasks import attempt_load_one_weight
                ck, _ckpt = attempt_load_one_weight(cand)
            finally:
                if _off is not None:
                    os.environ["YOLO_OFFLINE"] = _off
            cd = ck.state_dict(); md = model.model.state_dict()
            copied = sum(1 for k, v in cd.items()
                         if k in md and md[k].shape == v.shape)
            for k, v in cd.items():
                if k in md and md[k].shape == v.shape:
                    md[k].copy_(v)
            print(f"  [pretrained] COCO init from {cand}: copied {copied} tensors")
        except Exception as e:
            print("  [pretrained] FAILED to load", pretrained, "->", e,
                  "| continuing from random init")
    else:
        print("  [pretrained] NONE - training from scratch")
    n=sum(p.numel() for p in model.model.parameters())
    print(f"built RapierNet yolo11-{scale} + SARF/CoordGate/DySample  ({n/1e6:.2f}M params)")
    return model, yml

# ========================== STEP 2 : TRAIN (SESSION-SAFE) =====================
def run_train(data_yaml, epochs, imgsz, batch, run_name, lr0=0.01, lrf=0.01,
              warmup_epochs=3.0, close_mosaic=10, cls=0.8, seed_pt=None,
              pretrained=None):
    """Train (or resume) run_name to `epochs` total. Returns best.pt path.
    seed_pt: on FRESH start, initialize from these weights (e.g. stage-1 best.pt
    for the high-res fine-tune). Resume is auto-detected from the run's last.pt."""
    run_dir = os.path.join(WORKDIR, run_name)
    last_pt = os.path.join(run_dir, "weights", "last.pt")
    ckpt_args = dict(data=data_yaml, epochs=epochs, imgsz=imgsz, batch=batch,
                     device=0 if torch.cuda.is_available() else "cpu",
                     workers=min(6, os.cpu_count() or 4), seed=42,
                     deterministic=False, cache=False, patience=40,
                     project=WORKDIR, name=run_name, exist_ok=True,
                     save_period=SAVE_PERIOD, lr0=lr0, lrf=lrf,
                     optimizer="SGD", momentum=0.937, weight_decay=0.0005,
                     warmup_epochs=warmup_epochs, cos_lr=True,
                     close_mosaic=close_mosaic, amp=True, val=True, plots=False,
                     mosaic=1.0, mixup=0.15, copy_paste=0.2, degrees=10.0,
                     perspective=0.0001, flipud=0.1, fliplr=0.5,
                     hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, scale=0.5,
                     translate=0.1, erasing=0.3, cls=cls, box=7.5, dfl=1.5)
    if os.path.isfile(last_pt):
        # Only resume if the checkpoint is genuinely resumable (epoch + optimizer
        # present). If it was a naturally-finished run (optimizer stripped), resume
        # would silently restart in the wrong default dir - so fall back to seeding
        # from this run's best.pt (or abort to let the caller re-run stage).
        _rk = torch.load(last_pt, map_location="cpu", weights_only=False)
        _resumable = (_rk.get("epoch", -1) >= 0) and (_rk.get("optimizer") is not None)
        del _rk; gc.collect()
        if _resumable:
            print("[resume] continuing run:", last_pt)
            model = YOLO(last_pt)
            model.train(resume=True, data=data_yaml,
                        device=ckpt_args["device"], batch=ckpt_args["batch"],
                        workers=ckpt_args["workers"], imgsz=imgsz,
                        save_period=SAVE_PERIOD)
            return os.path.join(run_dir, "weights", "best.pt")
        else:
            _best = os.path.join(run_dir, "weights", "best.pt")
            print("[warn] last.pt not resumable (finished run). Seeding from best.pt if available...")
            if os.path.isfile(_best):
                seed_pt = _best
            else:
                raise RuntimeError(f"Run {run_name} has a non-resumable last.pt and no best.pt.")
    if seed_pt is not None:
        print("[fresh] initializing from seed weights:", seed_pt)
        model = YOLO(seed_pt)
        model.train(**ckpt_args)
    else:
        model, _ = build_rapiernet(MODEL_SCALE, pretrained=pretrained)
        model.train(**ckpt_args)
    return os.path.join(run_dir, "weights", "best.pt")

def train_epoch_progress(run_name):
    """Return (trained_epochs, status) for run_name's run dir.
    status: 'running' (mid-run, optimizer present) | 'finished' (epoch=-1,
    optimizer stripped) | 'best-only' | 'none'.
    Finished ultralytics runs save last.pt with epoch=-1 and NO optimizer;
    returning -1 would make main_pipeline retrain the whole stage, so we map
    a finished run to (epochs-1, 'finished')."""
    def _ep(ck):
        ep = ck.get("epoch", -1)
        tr = ck.get("train_args", {})
        n = int(tr.get("epochs", 0)) if isinstance(tr, dict) else 0
        if ep is None or ep < 0:
            return (n - 1 if n else -1), "finished"
        return ep, "running"
    last_pt = os.path.join(WORKDIR, run_name, "weights", "last.pt")
    if os.path.isfile(last_pt):
        ck = torch.load(last_pt, map_location="cpu", weights_only=False)
        ep, st = _ep(ck); del ck; gc.collect()
        return ep, st
    best_pt = os.path.join(WORKDIR, run_name, "weights", "best.pt")
    if os.path.isfile(best_pt):
        ck = torch.load(best_pt, map_location="cpu", weights_only=False)
        ep, st = _ep(ck); del ck; gc.collect()
        return ep, "best-only" if st == "running" else st
    return -1, "none"

def prepare_dataset():
    """Build clean 4-class set once; reuse if already built (saves session time)."""
    out = "/kaggle/working/rdd4"
    data_yaml = "/kaggle/working/rdd4.yaml"
    marker = os.path.join(out, ".built")
    if os.path.isfile(marker):
        print("[Step 0] reusing cached clean dataset (", out, ")")
        return data_yaml
    print("[Step 0] Building clean 4-class dataset + honest split ...")
    data_yaml, _ = remap_and_split()
    with open(marker, "w") as f:
        f.write(time.ctime())
    return data_yaml

# ========================== STEP 3 : EVAL + WBF + TTA ========================
def eval_model(best_pt, data_yaml, tta=True):
    DEV = 0 if torch.cuda.is_available() else "cpu"
    m = YOLO(best_pt)
    met = m.val(data=data_yaml, conf=0.25, iou=0.5, imgsz=960, device=DEV, batch=8)
    f1 = 2*met.box.mp*met.box.mr/(met.box.mp+met.box.mr+1e-9)
    print("\nFINAL  mAP50=%.3f  mAP50-95=%.3f  P=%.3f  R=%.3f  F1=%.3f" % (
        met.box.map50, met.box.map, met.box.mp, met.box.mr, f1))
    try:
        import yaml as _y
        _names = _y.safe_load(open(data_yaml))["names"]
        if isinstance(_names, dict):
            _names = [_names[i] for i in range(len(_names))]
    except Exception:
        _names = [str(i) for i in range(met.box.nc)]
    for i in range(min(met.box.nc, len(met.box.ap))):
        print("   %-4s mAP50=%.3f mAP50-95=%.3f" % (_names[i], met.box.ap50[i], met.box.ap[i]))
    if tta:
        met_t = m.val(data=data_yaml, conf=0.25, iou=0.5, imgsz=960, device=DEV,
                      batch=8, augment=True)
        f1_t = 2*met_t.box.mp*met_t.box.mr/(met_t.box.mp+met_t.box.mr+1e-9)
        print("TTA    mAP50=%.3f  mAP50-95=%.3f  P=%.3f  R=%.3f  F1=%.3f" % (
            met_t.box.map50, met_t.box.map, met_t.box.mp, met_t.box.mr, f1_t))
    if HAS_WBF:
        try:
            import ensemble_boxes as eb
            holdout = glob.glob("/kaggle/working/rdd4/val/images/*.jpg")
            if not holdout:
                holdout = glob.glob("/kaggle/working/rdd4/val/images/*.png")
            holdout = holdout[:40]
            lo = m.predict(source=holdout, conf=0.10, iou=0.5, imgsz=960, device=DEV, verbose=False)
            hi = m.predict(source=holdout, conf=0.25, iou=0.5, imgsz=960, device=DEV, verbose=False)
            total = 0
            for a, b in zip(lo, hi):
                bx, sc, lb = [], [], []
                for r in (a, b):
                    if r.boxes is None or len(r.boxes) == 0:
                        continue
                    bx.append((r.boxes.xyxy/960).cpu().numpy().tolist())
                    sc.append(r.boxes.conf.cpu().numpy().tolist())
                    lb.append(r.boxes.cls.int().cpu().numpy().tolist())
                total += len(eb.nms_wbf(bx, sc, lb, iou_thr=0.55, weights=None)) if len(bx) >= 2 else (len(bx[0]) if bx else 0)
            print("WBF sample over %d imgs -> %d fused boxes" % (min(40, len(holdout)), total))
        except Exception as e:
            print("   WBF demo skipped:", e)

# ========================== ORCHESTRATION ====================================
def adaptive_batch(per_img_scale=1.0):
    """Pick a batch that fits VRAM. 640px yolo11m ~ 24/16GB GPU; 960px ~ 1/3.
    x2 GPUs: returns the per-GPU batch; ultralytics auto-accumulates to nbs=64."""
    if torch.cuda.is_available():
        try:
            mem = torch.cuda.get_device_properties(0).total_memory / 1e9
            base = 24 if mem >= 14 else 12          # per 16GB GPU @640px
            batch = int(max(4, base * per_img_scale))
            return batch, torch.cuda.device_count()
        except Exception:
            return 16, 1
    return 4, 0

def main_pipeline(force_stage2=False):
    restore_from_input()
    data_yaml = prepare_dataset()

    # ---- Stage 1: main 640px training (resumable) ----
    ep1, st1 = train_epoch_progress(DATA_NAME)
    print(f"\n[Stage1] last.pt epoch={ep1}  (target {EPOCHS_STAGE1})")
    if ep1 < EPOCHS_STAGE1 - 1:
        batch_s1, ngpu = adaptive_batch(1.0)
        print("  starting/resuming stage1  batch=%d on %d GPU(s) @640px" % (batch_s1, max(1, ngpu)))
        best1 = run_train(data_yaml, EPOCHS_STAGE1, 640, batch_s1, DATA_NAME,
                          lr0=0.01, lrf=0.01, warmup_epochs=3.0, close_mosaic=10,
                          cls=0.8, pretrained=PRETRAINED)
        ep1, _ = train_epoch_progress(DATA_NAME)

    s1_best = os.path.join(WORKDIR, DATA_NAME, "weights", "best.pt")

    # ---- Stage 2: 960px high-res fine-tune (resumable) ----
    s2_name = DATA_NAME + "_hr"
    s2_has_started = os.path.isfile(os.path.join(WORKDIR, s2_name, "weights", "last.pt"))
    ep2, _ = train_epoch_progress(s2_name)
    print(f"\n[Stage2] last.pt epoch={ep2}  (target {EPOCHS_STAGE2})")
    # Discard an untrained/raw stage-2 run (<=1 epoch): it may carry a stale
    # seed and a stale epoch budget, and ultralytics resume=True would use the
    # checkpoint's old epochs instead of our EPOCHS_STAGE2. Fresh-seed instead.
    if s2_has_started and ep2 <= 1:
        print(f"[Stage2] existing {s2_name} is raw/untrained (epoch={ep2}); "
              "discarding to seed fresh from stage-1 best")
        shutil.rmtree(os.path.join(WORKDIR, s2_name), ignore_errors=True)
        s2_has_started = False
        ep2, _ = train_epoch_progress(s2_name)
    stage1_done = (ep1 >= EPOCHS_STAGE1 - 1)
    if stage1_done and (force_stage2 or ep2 < EPOCHS_STAGE2 - 1):
        batch_s2, _ = adaptive_batch(0.35)
        print("  stage2 batch=%d @960px  (seed=%s, resume=%s)"
              % (max(4, batch_s2),
                 "stage1-best" if (os.path.isfile(s1_best) and not s2_has_started) else "n/a",
                 s2_has_started))
        best2 = run_train(data_yaml, EPOCHS_STAGE2, 960, max(4, batch_s2), s2_name,
                          lr0=0.002, lrf=0.1, warmup_epochs=1.0, close_mosaic=5, cls=0.8,
                          seed_pt=s1_best if (os.path.isfile(s1_best) and not s2_has_started) else None)
        ep2, _ = train_epoch_progress(s2_name)

    # ---- Final eval ----
    final_pt = os.path.join(WORKDIR, s2_name, "weights", "best.pt")
    if not os.path.isfile(final_pt):
        final_pt = s1_best
    print("\n[Eval] best checkpoint ->", final_pt)
    eval_model(final_pt, data_yaml, tta=True)
    print("\n" + "="*72)
    print("  RAPIERNET run complete. Best weights:", final_pt)
    print("="*72)

# ---- RESTORE CHECKPOINTS FROM UPLOADED INPUT DATASET (run at session start) --
# Kaggle mounts input datasets read-only under /kaggle/input and does NOT
# auto-extract/relocate them to /kaggle/working. Without this step, a fresh
# session would NOT find last.pt at /kaggle/working/rapiernet/... and would
# silently restart at epoch 0. This copies the packed run dir back so resume works.
def restore_from_input():
    """Rehydrate /kaggle/working from uploaded checkpoints.

    Supports layouts found under /kaggle/input:
      A) pack layout : <root>/rapiernet/weights/{last,best}.pt  (+ _hr run)
      B) flat layout : <dir>/{last,best}.pt + {rapiernet_m.yaml,results.csv}
      C) zips        : any .zip containing A or B, extracted first.
    Adopts runs, then prints resumability so the session continues where it
    left off instead of cold-restarting a finished stage."""
    base = WORKDIR
    if os.path.isdir(os.path.join(base, DATA_NAME, "weights")):
        print(f"[RESUME] {DATA_NAME} already present at {base} - no restore needed")
        return True

    def _is_ours(path):
        return "rapiernet" in path.lower() or "stage2" in path.lower()

    import zipfile
    for z in sorted(set(glob.glob("/kaggle/input/**/*.zip", recursive=True))):
        try:
            with zipfile.ZipFile(z) as zf:
                if any(nm.lower().endswith("last.pt") for nm in zf.namelist()):
                    print("[RESUME] extracting", z)
                    zf.extractall(base)
        except Exception as e:
            print("[RESUME] skipped zip", z, "->", e)

    adopted = set()
    cands = sorted(set(glob.glob("/kaggle/input/**/last.pt", recursive=True)) |
                   set(glob.glob(os.path.join(base, "**", "last.pt"), recursive=True)))
    for last in cands:
        if not _is_ours(last):
            continue
        parent = os.path.dirname(last)
        if os.path.basename(parent) == "weights":
            run_root = os.path.dirname(parent)
            run_name = os.path.basename(run_root)
            if run_name in adopted:
                continue
            if not os.path.isdir(os.path.join(base, run_name)):
                print(f"[RESUME] copying run dir {run_name} <- {run_root}")
                shutil.copytree(run_root, os.path.join(base, run_name))
            adopted.add(run_name)
        else:
            run_name = DATA_NAME + ("_hr" if os.path.basename(parent).endswith("_hr") else "")
            if run_name in adopted:
                continue
            dst_w = os.path.join(base, run_name, "weights")
            os.makedirs(dst_w, exist_ok=True)
            for fn in os.listdir(parent):
                srcp = os.path.join(parent, fn)
                if not os.path.isfile(srcp):
                    continue
                if fn.lower().endswith(".pt"):
                    shutil.copy2(srcp, os.path.join(dst_w, fn))
                else:
                    os.makedirs(os.path.join(base, run_name), exist_ok=True)
                    shutil.copy2(srcp, os.path.join(base, run_name, fn))
            print(f"[RESUME] flattened {parent} -> {base}/{run_name}")
            adopted.add(run_name)

    for nm in adopted:
        ep, st = train_epoch_progress(nm)
        if ep >= 0 and st == "running":
            print(f"[RESUME] {nm}: epoch={ep}, resumable (optimizer present)")
        elif ep >= 0:
            print(f"[RESUME] {nm}: finished at epoch={ep} -> will seed from best.pt / skip")
        else:
            print(f"[RESUME] {nm}: state unclear ({st})")
    if not adopted:
        print("[RESUME] no rapiernet checkpoints found under /kaggle/input "
              "(cold start from random init)")
    return bool(adopted)
def pack_next_session():
    z = "/kaggle/working/rapiernet_continue.zip"
    base = WORKDIR
    targets = []
    for nm in (DATA_NAME, DATA_NAME + "_hr"):
        d = os.path.join(base, nm)
        if os.path.isdir(d):
            targets.append(d)
    import zipfile
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
        for d in targets:
            for root, _, files in os.walk(d):
                for fn in files:
                    p = os.path.join(root, fn)
                    zf.write(p, os.path.relpath(p, base))
    print("Packed checkpoint(s) for next session ->", z)
    print("Upload", z, "to Kaggle as a dataset, extract into /kaggle/working,")
    print("then re-run this notebook (in the same /kaggle/working layout) to continue.")

if __name__ == "__main__":
    try:
        main_pipeline()
    except Exception:
        import traceback; traceback.print_exc()
        print("\n[FATAL] aborted")



  RAPIERNET-RDD2022  (scale=m)  ->  target: beat StripRFNet
Python 3.12.13 | torch 2.10.0+cu128 | CUDA True
  GPU: Tesla T4  15.6 GB  x2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 5.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
[RAPIERNET] custom modules registered (cross-session unpickle ready)
[RESUME] flattened /kaggle/input/datasets/amlmoawad/rdd2022-stage2-checkpoints/rapiernet_continue/rapiernet_continue -> /kaggle/working/rapiernet
[RESUME] flattened /kaggle/input/datasets/amlmoawad/rdd2022-stage2-checkpoints/rapiernet_hr/rapiernet_h

Traceback (most recent call last):
  File "/tmp/ipykernel_58/3169635311.py", line 629, in <cell line: 0>
    main_pipeline()
  File "/tmp/ipykernel_58/3169635311.py", line 515, in main_pipeline
    best2 = run_train(data_yaml, EPOCHS_STAGE2, 960, max(4, batch_s2), s2_name,
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_58/3169635311.py", line 356, in run_train
    model.train(resume=True, data=data_yaml,
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 871, in train
    self.trainer.train()
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.py", line 246, in train
    self._do_train()
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.py", line 653, in _do_train
    self.final_eval()
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.py", line 962, in final_eval
    self.metrics = self.validator(model=model)
                  

In [2]:
import torch
from ultralytics import YOLO

# التأكد من تسجيل الموديلات المخصصة قبل التحميل
register_custom_modules()

# مسار الأوزان النهائية للمرحلة الثانية
best_pt_path = "/kaggle/working/rapiernet_hr/weights/best.pt"
yaml_path = "/kaggle/working/rdd4.yaml"

if os.path.isfile(best_pt_path):
    print("جاري تشغيل التقييم النهائي يدوياً...")
    model = YOLO(best_pt_path)
    
    # تشغيل الـ val بشكل آمن مع إيقاف الـ amp لتفادي خطأ الـ Half/Float
    metrics = model.val(
        data=yaml_path,
        imgsz=960,
        batch=8,
        device=0 if torch.cuda.is_available() else "cpu",
        amp=False  # حل مشكلة التعارض هنا
    )
    
    print("\n-----------------------------------------")
    print(f"النتيجة النهائية mAP50: {metrics.box.map50:.3f}")
    print(f"النتيجة النهائية mAP50-95: {metrics.box.map:.3f}")
    print("-----------------------------------------")
else:
    print("ملف الأوزان غير موجود، تأكدي من المسار.")

جاري تشغيل التقييم النهائي يدوياً...
Ultralytics 8.4.142 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rapiernet_m summary (fused): 117 layers, 48,709,925 parameters, 0 gradients, 90.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2111.6±603.0 MB/s, size: 125.2 KB)
val: Scanning /kaggle/working/rdd4/val/labels.cache... 1835 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1835/1835 699.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 230/230 1.4it/s 2:460.6ss
                   all       1835       4572      0.667       0.59      0.636      0.335
                   D00        627        787      0.712      0.613      0.688      0.358
                   D10        513        779      0.622      0.539       0.59      0.286
                   D20        911       1780      0.629      0.557      0.576      0.314
                   D40        728       1226      0.703      0.652      0.692

In [3]:
pack_next_session()

Packed checkpoint(s) for next session -> /kaggle/working/rapiernet_continue.zip
Upload /kaggle/working/rapiernet_continue.zip to Kaggle as a dataset, extract into /kaggle/working,
then re-run this notebook (in the same /kaggle/working layout) to continue.
